# Preprocesamiento del dataset COMPAS

<!-- Este notebook:
1. Carga el dataset COMPAS.
2. Aplica el preprocesamiento visto hasta ahora.
3. Genera archivos `.csv` listos para usar en entrenamiento.
4. Conserva un contexto de auditoría con los grupos protegidos originales.

Incluye:
- limpieza básica,
- filtrado de filas inválidas,
- selección de los subconjuntos de 2, 7 y 8 características,
- codificación de variables categóricas,
- guardado de archivos finales y del contexto para Aequitas. -->


In [1]:
# Dependencias

from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder


## 1. Configuración

In [ ]:
RAW_DIR = Path("data") / "raw"
PROCESSED_DIR = Path("data") / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Nombre esperado del archivo original
RAW_FILE = RAW_DIR / "compas-scores-two-years.csv"

# Salidas
PROCESSED_FILE = PROCESSED_DIR / "compas_preprocessed.csv"
AUDIT_CONTEXT_FILE = PROCESSED_DIR / "compas_audit_context.csv"


## 2. Carga del dataset

In [3]:
def locate_csv(raw_dir: Path) -> Path:
    candidates = sorted(raw_dir.rglob("*.csv"))
    if not candidates:
        raise FileNotFoundError(
            f"No encontré CSV en {raw_dir}. Coloca ahí 'compas-scores-two-years.csv'."
        )
    for p in candidates:
        if p.name.lower() == "compas-scores-two-years.csv":
            return p
    return candidates[0]

if RAW_FILE.exists():
    csv_path = RAW_FILE
else:
    csv_path = locate_csv(RAW_DIR)

df = pd.read_csv(csv_path)
print("Archivo cargado:", csv_path)
print("Forma original:", df.shape)
display(df.head())


Archivo cargado: data/raw/compas-scores-two-years.csv
Forma original: (7214, 53)


,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


## 3. Inspección rápida

In [4]:
print("Columnas:", len(df.columns))
display(pd.DataFrame({
    "columna": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "nulos": [int(df[c].isna().sum()) for c in df.columns],
}).head(25))

print("\nMuestras por raza:")
if "race" in df.columns:
    display(df["race"].value_counts(dropna=False))


Columnas: 53


,columna,dtype,nulos
0,id,int64,0
1,name,object,0
2,first,object,0
3,last,object,0
4,compas_screening_date,object,0
5,sex,object,0
6,dob,object,0
7,age,int64,0
8,age_cat,object,0
9,race,object,0



Muestras por raza:


race
African-American    3696
Caucasian           2454
Hispanic             637
Other                377
Asian                 32
Native American       18
Name: count, dtype: int64

## 4. Limpieza 

In [5]:
def clean_compas(raw: pd.DataFrame) -> pd.DataFrame:
    data = raw.copy()
    
    mask = (
        (df["days_b_screening_arrest"] <= 30)
        & (df["days_b_screening_arrest"] >= -30)
        & (df["is_recid"] != -1)
        & (df["c_charge_degree"] != "O")
        & (df["score_text"] != "N/A")
    )

    data = data[mask].copy()

    return data

clean_df = clean_compas(df)
print("Forma después de limpiar:", clean_df.shape)

nulls = clean_df.isna().sum().sort_values(ascending=False)
display(nulls[nulls > 0].to_frame("nulos"))


Forma después de limpiar: (6172, 53)


,nulos
violent_recid,6172
vr_charge_degree,5480
vr_offense_date,5480
vr_case_number,5480
vr_charge_desc,5480
c_arrest_date,5388
r_jail_in,4175
r_days_from_arrest,4175
r_jail_out,4175
r_charge_desc,3228


## 5. Selección de variables

In [ ]:
FEATURES_2 = ["age", "priors_count"]

FEATURES_7 = [
    "sex",
    "age",
    "juv_fel_count",
    "juv_misd_count",
    "priors_count",
    "c_charge_degree",
]

FEATURES_8 = FEATURES_7 + ["race"]
TARGET = "two_year_recid"

required_cols = sorted(set(FEATURES_8 + [TARGET]))
missing = [c for c in required_cols if c not in clean_df.columns]
if missing:
    print("Columnas faltantes:", missing)
else:
    print("Todas las columnas necesarias están presentes.")


Todas las columnas necesarias están presentes.


## 6. Codificación de variables categóricas

In [8]:
def encode_for_model(data: pd.DataFrame, feature_cols: list[str], target_col: str = TARGET) -> pd.DataFrame:
    frame = data.copy()

    needed = feature_cols + ([target_col] if target_col in frame.columns else [])
    frame = frame.loc[:, [c for c in needed if c in frame.columns]].copy()

    cat_cols = [c for c in feature_cols if frame[c].dtype == "object" or str(frame[c].dtype).startswith("category")]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    for col in num_cols:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
        frame[col] = frame[col].fillna(frame[col].median())

    for col in cat_cols:
        frame[col] = frame[col].fillna("Unknown").astype(str)

    if cat_cols:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        frame[cat_cols] = enc.fit_transform(frame[cat_cols]).astype(int)

    if target_col in frame.columns:
        frame[target_col] = pd.to_numeric(frame[target_col], errors="coerce")
        frame = frame.dropna(subset=[target_col]).copy()
        frame[target_col] = frame[target_col].astype(int)

    frame = frame.dropna().reset_index(drop=True)
    return frame

df2 = encode_for_model(clean_df, FEATURES_2)
df7 = encode_for_model(clean_df, FEATURES_7)
df8 = encode_for_model(clean_df, FEATURES_8)

print("2 features:", df2.shape)
print("7 features:", df7.shape)
print("8 features:", df8.shape)
display(df8.head())


2 features: (6172, 3)
7 features: (6172, 7)
8 features: (6172, 8)


,sex,age,juv_fel_count,juv_misd_count,priors_count,c_charge_degree,race,two_year_recid
0,1,69,0,0,0,0,5,0
1,1,34,0,0,0,0,0,1
2,1,24,0,0,4,0,0,1
3,1,44,0,0,0,1,5,0
4,1,41,0,0,14,0,2,1


## 8. Guardado de archivos y contexto de auditoría


In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# df2.to_csv(PROCESSED_DIR / "compas_features_2.csv", index=False)
# df7.to_csv(PROCESSED_DIR / "compas_features_7.csv", index=False)
df8.to_csv(PROCESSED_FILE, index=False)

# Contexto sin codificar para auditorías de equidad. Su orden debe coincidir
# exactamente con el de los tres CSV usados para entrenar los modelos.
audit_context = (
    clean_df.loc[:, ["id", "race", "sex", "age_cat", TARGET]]
    .copy()
    .rename(columns={"id": "entity_id", TARGET: "label_value"})
    .reset_index(drop=True)
)

for dataset_name, model_df in {
    "compas_features_2.csv": df2,
    "compas_features_7.csv": df7,
    "compas_preprocessed.csv": df8,
}.items():
    if len(model_df) != len(audit_context):
        raise ValueError(
            f"{dataset_name} no está alineado con el contexto de auditoría. "
            "No se puede generar un archivo fiable para Aequitas."
        )
    if not np.array_equal(
        model_df[TARGET].to_numpy(), audit_context["label_value"].to_numpy()
    ):
        raise ValueError(
            f"Las etiquetas de {dataset_name} no coinciden con el contexto de auditoría."
        )

audit_context.to_csv(AUDIT_CONTEXT_FILE, index=False)

print("Archivos generados:")
print(PROCESSED_DIR / "compas_features_2.csv")
print(PROCESSED_DIR / "compas_features_7.csv")
print(PROCESSED_FILE)
print(AUDIT_CONTEXT_FILE)


Archivos generados:
data/processed/compas_features_2.csv
data/processed/compas_features_7.csv
data/processed/compas_preprocessed.csv
data/processed/compas_audit_context.csv


## 9. Verificación final


In [10]:
print("Vista previa del CSV final:")
display(pd.read_csv(PROCESSED_FILE).head())

print("\nDistribución del target en el conjunto final:")
display(pd.read_csv(PROCESSED_FILE)[TARGET].value_counts(normalize=True).rename("proporción"))

print("\nVista previa del contexto de auditoría Aequitas:")
display(pd.read_csv(AUDIT_CONTEXT_FILE).head())


Vista previa del CSV final:


,sex,age,juv_fel_count,juv_misd_count,priors_count,c_charge_degree,race,two_year_recid
0,1,69,0,0,0,0,5,0
1,1,34,0,0,0,0,0,1
2,1,24,0,0,4,0,0,1
3,1,44,0,0,0,1,5,0
4,1,41,0,0,14,0,2,1



Distribución del target en el conjunto final:


two_year_recid
0    0.54488
1    0.45512
Name: proporción, dtype: float64


Vista previa del contexto de auditoría Aequitas:


,entity_id,race,sex,age_cat,label_value
0,1,Other,Male,Greater than 45,0
1,3,African-American,Male,25 - 45,1
2,4,African-American,Male,Less than 25,1
3,7,Other,Male,25 - 45,0
4,8,Caucasian,Male,25 - 45,1
